In [ ]:
'''
This notebook finds Sentinel-2 L1B granules and sets up processing of the
Sen2VM algorithm, to generate geolocation information, on each granule.


Please note: This notebook is a heavily modified version of the
Sen2VM example notebook, which is available on github here:
https://github.com/sen2vm/sen2vm-core/blob/main/sen2vm-notebook/src/notebook.ipynb

Depending on your needs and objectives, it may be better to start directly
from the Sen2VM example for your own processing.

This script was originally intended for internal use and may have
limited functionality, bugs and other problems. It does not have 
detailed documentation. Please contact the ESA Sentinel-2 Next Generation 
mission scientist for more information: simon{dot}proud{at}esa{dot}int

Please also note, some directory structures in this notebook are
hardcoded and will require modification. Check the whole notebook
carefully before running on your system.

IMPORTANT NOTE: This script generates a docker_run file that will actually
run the Sen2VM processing. The script itself *does not* do any processing.

Once the script is complete, run the `docker_run` file from your terminal.
'''

from datetime import datetime
from tqdm import tqdm
from glob import glob
import requests
import shutil
import json
import re
import os

def int_to_roman(n):
    vals = [
        (1000, 'm'), (900, 'cm'), (500, 'd'), (400, 'cd'),
        (100, 'c'), (90, 'xc'), (50, 'l'), (40, 'xl'),
        (10, 'x'), (9, 'ix'), (5, 'v'), (4, 'iv'), (1, 'i')
    ]
    res = ""
    for v, s in vals:
        while n >= v:
            res += s
            n -= v
    return res

def sort_gipp(WORKDIR, verbose):
    gipp_dir = os.path.join(WORKDIR, "DATA")     
    os.makedirs(gipp_dir, exist_ok=True)             

    gipp_repo_name = "sen2vm-gipp-database"
    gipp_repo_path = os.path.join(gipp_dir, gipp_repo_name)

    if os.path.exists(gipp_repo_path):
        if verbose:
            print(f"Removing existing repository: {gipp_repo_path}")
        shutil.rmtree(gipp_repo_path)

    if verbose:
        print("Cloning sen2vm-gipp-database...")
    !git clone https://github.com/sen2vm/sen2vm-gipp-database.git {gipp_repo_path}
    if verbose:
        print("Clone complete.\n")
    if verbose:
        print("GIPP processing finished successfully.")
    
def sort_iers(WORKDIR, PATH_L1B_DATA, verbose):
    DATA_DIR = os.path.join(WORKDIR, "DATA")

    if not os.path.exists(DATA_DIR):
        raise RuntimeError(f"DATA directory not found: {DATA_DIR}")

    if verbose:
        print("Bulletin output directory:", DATA_DIR)

    datastrip_dir = os.path.join(PATH_L1B_DATA, "DATASTRIP")

    if not os.path.isdir(datastrip_dir):
        raise RuntimeError(f"DATASTRIP directory not found: {datastrip_dir}")

    datastrip_entries = os.listdir(datastrip_dir)
    if not datastrip_entries:
        raise RuntimeError(f"No DATASTRIP found in: {datastrip_dir}")

    # Take the first DATASTRIP product
    datastrip_name = datastrip_entries[0]

    match = re.search(r"_S(\d{8})T\d{6}_", datastrip_name)
    if not match:
        raise RuntimeError("Could not extract product date from DATASTRIP name.")

    product_date_str = match.group(1)

    year = int(product_date_str[:4])
    month = int(product_date_str[4:6])
    day = int(product_date_str[6:8])

    product_date = datetime(year, month, day)

    if verbose:
        print("Product date extracted from DATASTRIP:", product_date.date(), "\n")

    roman_year = int_to_roman(year - 1987)
    doy = product_date.timetuple().tm_yday
    index = (doy - 1) // 7 + 1
    if verbose:
        print(f"Bulletin A Roman year: {roman_year}")
        print(f"Initial weekly index: {index}\n")

    found = False

    while index > 0:
        index_str = f"{index:03d}"
        url = f"https://datacenter.iers.org/data/6/bulletina-{roman_year}-{index_str}.txt"
        if verbose:
            print("Trying bulletin:", url)

        response = requests.get(url)

        if response.status_code == 200:
            if verbose:
                print("Bulletin found:", index_str)
            dest_file = os.path.join(DATA_DIR, f"bulletina-{roman_year}-{index_str}.txt")
            found = True
            break

        index -= 1

    if not found:
        raise RuntimeError("No Bulletin A available for current or previous weeks.")

    with open(dest_file, "wb") as f:
        f.write(response.content)
        
    if verbose:    
        print("Downloaded:", dest_file)
    return dest_file

def make_config(WORKDIR, PATH_L1B_DATA, datestr, iers_b_file, verbose):
    USERCONF_DIR = os.path.join(WORKDIR, "UserConf")
    os.makedirs(USERCONF_DIR, exist_ok=True)

    if verbose:
        print("UserConf directory:", USERCONF_DIR)

    GEOID_DIR = os.path.join(WORKDIR, "DATA", "GEOID")
    os.makedirs(GEOID_DIR, exist_ok=True)

    if verbose:
        print("Geoid directory:", GEOID_DIR)

    # =====================================================
    # 1. Extract mission from DATASTRIP
    # =====================================================

    datastrip_dir = os.path.join(PATH_L1B_DATA, "DATASTRIP")

    if not os.path.isdir(datastrip_dir):
        raise RuntimeError(f"DATASTRIP directory not found: {datastrip_dir}")

    datastrip_entries = os.listdir(datastrip_dir)
    if not datastrip_entries:
        raise RuntimeError(f"No DATASTRIP found in: {datastrip_dir}")

    datastrip_name = datastrip_entries[0]

    match = re.match(r"(S2[A-C])_OPER_", datastrip_name)
    if not match:
        raise RuntimeError("Cannot extract mission (S2A/S2B/S2C) from DATASTRIP name.")

    mission = match.group(1)


    # =====================================================
    # 2. Docker paths inside /workspace
    # =====================================================

    safe_name = os.path.basename(PATH_L1B_DATA)

    docker_l1b  = f"/{WORKDIR}/DATA/{safe_name}"
    docker_dem  = f"/{WORKDIR}/DATA/DEM"
    docker_gipp = f"/{WORKDIR}/DATA/sen2vm-gipp-database/{mission}"

    # =====================================================
    # 3. Geoid management
    # =====================================================
    # Relative path to DEM_GEOID from notebook
    internal_geoid_dir = f"/{WORKDIR}/DATA/DEM_GEOID/"

    # If WORKDIR/DATA/GEOID is empty -> copy internal files
    if len(os.listdir(GEOID_DIR)) == 0:
        if verbose:
            print("GEOID directory is empty -> copying default geoid files...")
        for f in os.listdir(internal_geoid_dir):
            src = os.path.join(internal_geoid_dir, f)
            dst = os.path.join(GEOID_DIR, f)
            shutil.copy(src, dst)

    # Detect .gtx inside GEOID_DIR
    geoid_files = [f for f in os.listdir(GEOID_DIR) if f.lower().endswith(".gtx")]
    if len(geoid_files) == 0:
        raise RuntimeError("No .gtx geoid file found in WORKDIR/DATA/GEOID.")

    docker_geoid = f"/data/DATA/GEOID/{geoid_files[0]}"

    # =====================================================
    # 5. Build config dictionary
    # =====================================================

    config = {
        "l1b_product": docker_l1b,
        "gipp_folder": docker_gipp,
        "auto_gipp_selection": True,
        "grids_overwriting": True,
        "dem": docker_dem,
        "geoid": docker_geoid,
        "iers": iers_b_file,
        "operation": GRID_MODE,
        "deactivate_available_refining": False,
        "steps": {
            "10m_bands": STEPS["10m_bands"],
            "20m_bands": STEPS["20m_bands"],
            "60m_bands": STEPS["60m_bands"]
        },
        "export_alt": True
    }

    # =====================================================
    # Save config.json
    # =====================================================

    config_path = os.path.join(USERCONF_DIR, f"config_{datestr}.json")

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)

    if verbose:
        print(f"Configuration file generated: {config_path}")
    return config_path

def make_paramfile(WORKDIR, PATH_L1B_DATA, datestr, verbose):
    USERCONF_DIR = os.path.join(WORKDIR, "UserConf")
    os.makedirs(USERCONF_DIR, exist_ok=True)

    if verbose:
        print("UserConf directory:", USERCONF_DIR)

    # =====================================================
    # Locate GRANULE folders
    # =====================================================
    GR_TARGET_DIR = os.path.join(PATH_L1B_DATA, "GRANULE")

    if not os.path.exists(GR_TARGET_DIR):
        raise RuntimeError("GRANULE directory not found inside L1B SAFE.")

    granule_folders = [
        os.path.join(GR_TARGET_DIR, d)
        for d in os.listdir(GR_TARGET_DIR)
        if os.path.isdir(os.path.join(GR_TARGET_DIR, d))
    ]

    if verbose:
        print("Found", len(granule_folders), "granule folders.")

    # =====================================================
    # Extract detectors and bands from JP2
    # =====================================================
    detectors = set()
    bands = set()

    pattern = r"_D(\d+)_B(\d{1,2}[A]?)\.jp2$"

    for granule in granule_folders:
        img_data_dir = os.path.join(granule, "IMG_DATA")

        if not os.path.isdir(img_data_dir):
            continue

        for fname in os.listdir(img_data_dir):
            match = re.search(pattern, fname)
            if match:
                detectors.add(match.group(1))
                bands.add(f"B{match.group(2)}")

    detectors = sorted(detectors)
    bands = sorted(bands)

    if verbose:
        print("Detected detectors:", detectors)
        print("Detected bands:", bands)

    # =====================================================
    # Write params.json
    # =====================================================

    # Validate keep_detectors: must be a non-empty list (user requirement)
    if not isinstance(ORTHO_SETTINGS.get("keep_detectors"), list) or len(ORTHO_SETTINGS["keep_detectors"]) == 0:
        raise RuntimeError("ORTHO_SETTINGS['keep_detectors'] must be a non-empty list of detector ids (e.g. ['01','02']).")

    # Filter detectors to only include those selected by the user
    keep_detectors_list = ORTHO_SETTINGS["keep_detectors"]
    selected_detectors = [d for d in keep_detectors_list if d in detectors]

    # Only include selected detectors and bands in params.json
    params = {
        "detectors": sorted(selected_detectors),
        "bands": sorted(ORTHO_SETTINGS["keep_bands"])
    }

    params_path = os.path.join(USERCONF_DIR, f"params_{datestr}.json")

    with open(params_path, "w") as f:
        json.dump(params, f, indent=4)

    if verbose:
        print("params.json written to:", params_path)
        print(f"  - Detectors: {params['detectors']}")
        print(f"  - Bands: {params['bands']}")
    
    return params_path

In [ ]:
# Start of processing code

# Workdir is the top level directory, here containing both the S2 data in a /DATA/ subdirectory and other required datasets
# such as the DEM, GIPP files, etc. Some of these files must be downloaded manually, like the DEM, and others are downloaded
# automatically as part of processing.
WORKDIR = "/data/"


# This is the file that actually contains the commands for running Sen2VM.
# Once the notebook is complete, run this file using your terminal (tested on bash / Ubuntu 22)
docker_runfile = f"{WORKDIR}/docker_run"

# For night processing, stick to direct mode
GRID_MODE = "direct"

# Switch to true for some additional log messages
verbose = False

# I have found these values work well, but other values may produce more
# accurate results, or indeed speed up processing at the expense of accuracy.
STEPS = {
    "10m_bands": 4.5,
    "20m_bands": 2.25,
    "60m_bands": 1
}

# If you only require specific bands or detectors, select them here.
ORTHO_SETTINGS = {
    "keep_detectors": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"],
    "keep_bands": ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09", "B10", "B11", "B12", "B8A"]
}

In [ ]:
# This script assumes that you L1B Sentinel-2 data is stored in subdirectories of `{workdir}/DATA/`

s2_l1b_acquisitions = glob(f"{WORKDIR}/DATA/S2*")
print(f"Found {len(s2_l1b_acquisitions)} acquisitions to process.")

counter = 0

# To speed up processing, multiple granules can be processed in parallel.
# This variable controls the number of parallel processes (default: 7).
# Adjust to suit your machine
n_parr = 7

# Get the GIPP files needed
sort_gipp(WORKDIR, verbose)

# Main processing loop
with open(docker_runfile, "w") as fid:
    for PATH_L1B_DATA in tqdm(s2_l1b_acquisitions): 
        # Hardcoded year (!)
        pos = PATH_L1B_DATA.rfind("2025")
        dtstr = PATH_L1B_DATA[pos:pos+15]

        iers_b_file = sort_iers(WORKDIR, PATH_L1B_DATA, verbose)
        config_file = make_config(WORKDIR, PATH_L1B_DATA, dtstr, iers_b_file, verbose)
        params_file = make_paramfile(WORKDIR, PATH_L1B_DATA, dtstr, verbose)
        
        cmd_run = f"docker run --rm -v {WORKDIR}:/data sen2vm"
        cmd_run = f"{cmd_run} -c {config_file}"
        cmd_run = f"{cmd_run} -p {params_file}"
        fid.write(cmd_run)
        counter += 1
        if counter < n_parr:
            fid.write(" & \n")
        else:
            fid.write("\n")
            counter = 0

Found 56 acquisitions to process.
Cloning into '/data/DATA/sen2vm-gipp-database'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 85 (delta 5), reused 79 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 17.51 MiB | 26.83 MiB/s, done.
Resolving deltas: 100% (5/5), done.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 56/56 [00:09<00:00,  5.62it/s]
